In [1]:
!pip install -U \
  "trl==0.17.0" \
  "transformers==4.51.3" \
  "peft==0.17.0" \
  "accelerate>=1.0.0" \
  "datasets>=3.1.0" \
  "bitsandbytes>=0.43.0" \
  sentencepiece

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
import json

import torch
from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
data_path = "drive/MyDrive/cs-224n-project/steering_subset.jsonl"
output_dir = "drive/MyDrive/cs-224n-project/mistral_dpo_lora"

test_prompt = "I'm 12 and stressed about homework. What should I do tonight?"


def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def make_dataset(path, tokenizer):
    training_data = load_data(path)

    data = [
        {
            "prompt": tokenizer.apply_chat_template(
              [{"role": "user", "content": training_prompt["unsafe_child"]}],
              tokenize=False,
              add_generation_prompt=True
            ),
            "chosen": training_prompt["safe_child"],
            "rejected": training_prompt["unsafe_adult"],
        }
        for training_prompt in training_data
    ]

    dataset = Dataset.from_list(data)
    train_test = dataset.train_test_split(test_size=0.02, seed=42)
    return DatasetDict({"train": train_test["train"], "test": train_test["test"]})


def main():
    set_seed(42)
    os.makedirs(output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    dataset = make_dataset(data_path, tokenizer)

    compute_dtype = torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config= BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        ),
        device_map="auto",
        torch_dtype=torch.float16
    )

    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    args = DPOConfig(
        output_dir=output_dir,
        num_train_epochs=1.0,
        learning_rate=1e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        logging_steps=10,
        save_steps=100,
        eval_steps=100,
        eval_strategy="steps",
        save_strategy="steps",
        save_total_limit=2,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        max_length=1024,
        max_prompt_length=512,
        beta=0.1,
        remove_unused_columns=False,
        report_to="none",
        bf16=False,
        fp16=True,
        optim="paged_adamw_8bit",
    )

    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        processing_class=tokenizer,
        peft_config=peft_config,
    )

    trainer.train()
    trainer.model.print_trainable_parameters()

    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": test_prompt}],
            tokenize=False,
            add_generation_prompt=True
        )
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response)


if __name__ == "__main__":
    main()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Extracting prompt in train dataset:   0%|          | 0/511 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/511 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/511 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

...

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss


trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


[INST] I'm 12 and stressed about homework. What should I do tonight? [/INST] I'm an AI language model and can't directly see or feel emotions, but I can give you some suggestions that might help you manage your stress about homework. Here are some steps you can take:

1. Prioritize: Make a list of all the homework assignments you have and rank them in order of importance or due date. This will help you focus on the most pressing tasks first.
2. Create a study schedule: Break up larger assignments into smaller, manageable tasks. Set a timer for each task to help you stay focused and take regular breaks.
3. Take care of yourself: Make sure you're eating a healthy dinner and getting enough rest. Exercise can also help reduce stress.
4. Ask for help: If you're feeling overwhelmed
